## Wstęp do Uczenia Maszynowego

#### Laboratorium 05


### Zakres laboratorium


- przygotowanie danych: wartości odstające i braki danych
- `Pipeline` i `ColumnTransformer`
- reprodukowalność podziałów danych i eksperymentów

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


In [ ]:
df = pd.read_csv("titanic.csv")
df

| Nazwa kolumny | Opis                                            |
|---------------|-------------------------------------------------|
| pclass        | Klasa biletu                                    |
| name          | Imię i nazwisko pasażera                        |
| sex           | Płeć pasażera                                   |
| age           | Wiek pasażera w latach                          |
| sibsp         | Liczba rodzeństwa / współmałżonków na pokładzie |
| parch         | Liczba rodziców / dzieci na pokładzie           |
| ticket        | Numer biletu                                    |
| fare          | Cena biletu (w funtach szterlingach z 1912 r.)  |
| cabin         | Numer kabiny                                    |
| embarked      | Port, w którym pasażer wsiadł na pokład         |
| boat          | Numer łodzi ratunkowej (dla ocalałych)          |
| body          | Numer odnalezionego ciała (dla zmarłych)        |
| home.dest     | Dom / Miejsce docelowe pasażera                 |
| survived      | Status przeżycia                                |

In [ ]:
y = df.survived
X = df.drop(["survived"], axis = 1)

### 1. Przygotowanie danych

#### Wartości odstające

Wartość skrajna nie musi być błędem. Najpierw wykrywamy takie obserwacje i sprawdzamy ich znaczenie w kontekście problemu; usuwamy je lub przekształcamy dopiero wtedy, gdy mamy ku temu uzasadnienie.


In [ ]:
# Rozkład opłaty za bilet. Boxplot wskazuje obserwacje daleko od pozostałych,
# ale sam nie rozstrzyga, czy są one błędami.
fig, ax = plt.subplots()
ax.boxplot(df["fare"].dropna(), orientation="horizontal")
ax.set_xlabel("fare")
ax.set_title("Rozkład opłaty za bilet")
plt.show()

q1, q3 = df["fare"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
fare_extreme = df.loc[df["fare"] > upper_fence, ["pclass", "fare", "name"]]
print(f"Próg 1,5 IQR: {upper_fence:.2f}")
fare_extreme.sort_values("fare", ascending=False).head()


W tym zbiorze wysokie wartości `fare` odpowiadają drogim biletom, więc nie traktujemy ich automatycznie jako błędów i ich nie usuwamy. Gdyby celem było modelowanie wrażliwe na silnie skośny rozkład, można byłoby później rozważyć np. transformację `log1p(fare)` $\Big(ln(1+x)\Big)$.

#### Braki danych

Braki można uzupełniać różnie zależnie od typu zmiennej, np. średnią dla cech numerycznych albo najczęstszą kategorią dla cech kategorycznych. Ważne, aby parametry imputacji wyznaczać tylko na danych treningowych — `Pipeline` zapewnia to automatycznie.


-----
##### *Zadanie 1*

Przyjrzyj się danym `titanic`. Korzystając z informacji o typach, brakach i rozkładach, podziel kolumny na: przydatne bez zmian, wymagające przygotowania oraz takie, które należy usunąć.


#### Decyzje dotyczące kolumn

- przydatne bez zmian:
- wymagające przygotowania:
- należy usunąć:

In [ ]:
# TODO: wybierz kolumny do usunięcia zgodnie z analizą powyżej.
# Zwróć szczególną uwagę na zmienne powodujące leakage: boat i body.
columns_to_drop = []
X = X.drop(columns=columns_to_drop)


-----
##### *Zadanie 2*

Przygotuj listy `numerical_features` i `categorical_features`, zawierające odpowiednio nazwy kolumn numerycznych i kategorycznych. Uruchom kod i sprawdź, czy odpowiadają decyzjom z poprzedniego zadania.


In [ ]:
# TODO: utwórz listy kolumn numerycznych i kategorycznych na podstawie ich typów.
numerical_features = []
categorical_features = []

print("Cechy numeryczne:", numerical_features)
print("Cechy kategoryczne:", categorical_features)


-----
##### *Zadanie 2 z gwiazdką*
Jak wykorzystując informację o typach kolumn podzielić je na numeryczne i kategoryczne?


#### Pipeline dla cech numerycznych

Dla cech numerycznych najpierw uzupełniamy braki średnią, a następnie standaryzujemy wartości. Drzewo nie wymaga skalowania, ale ten sam etap będzie można połączyć później z modelami wrażliwymi na skalę cech.


In [ ]:
numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])


-----
##### *Zadanie 3*

Przygotuj `Pipeline` dla zmiennych kategorycznych. Użyj `SimpleImputer()` ze strategią wyboru najczęstszej kategorii oraz `OneHotEncoder()` z ignorowaniem nieznanych kategorii.

https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html

In [ ]:
categorical_transformer = Pipeline([

    
])


#### Połączenie przygotowania danych z modelem

`ColumnTransformer` stosuje właściwy pipeline do każdej grupy kolumn. Następnie cały etap przygotowania łączymy z drzewem w jeden `Pipeline`.

https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html


In [ ]:
# TODO: połącz oba pipeline'y w ColumnTransformer.
preprocessor = ColumnTransformer([
    ("numerical", numerical_transformer, numerical_features),
    # TODO:
])

# TODO: połącz preprocessor z drzewem decyzyjnym w jeden Pipeline.
pipeline = Pipeline([
    
    
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
pipeline.fit(X_train, y_train)

print("Accuracy na zbiorze treningowym:", round(pipeline.score(X_train, y_train), 3))
print("Accuracy na zbiorze testowym:", round(pipeline.score(X_test, y_test), 3))


Jeżeli wynik na zbiorze treningowym jest wyraźnie wyższy niż na testowym, może to wskazywać na przeuczenie. W dalszej części laboratorium dostroimy złożoność drzewa za pomocą walidacji krzyżowej.


-----
##### *Zadanie 4*

Wczytaj zbiór danych `pima.csv`. Narysuj rozkład `BloodPressure` osobno dla klas `Outcome` i zastanów się, co mogą oznaczać wartości równe 0.


In [ ]:
pima = pd.read_csv("pima.csv")
y = pima["Outcome"]
X = pima.drop(columns="Outcome")


-----
##### *Zadanie 5*

a) Zastąp wartości 0 zmiennej `BloodPressure` brakami danych. Podziel dane na zbiór treningowy i testowy w proporcji 7:3. Przygotuj `Pipeline` z imputacją średnią i drzewem decyzyjnym. Oblicz `accuracy` na zbiorze treningowym i testowym.

b) Przeprowadź optymalizację hiperparametrów pipeline'u za pomocą `GridSearchCV` z 3-krotną walidacją krzyżową. Rozważ `max_depth` od 5 do 12 oraz `min_samples_leaf` od 3 do 9.

c) Podaj najlepsze parametry i najlepszą wartość accuracy w CV. Na zbiorze testowym porównaj accuracy oraz ROC AUC dostrojonego i bazowego pipeline'u.

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

https://scikit-learn.org/stable/modules/grid_search.html

In [ ]:
# TODO: zastąp zera w BloodPressure wartością np.nan.


In [ ]:
pipeline = Pipeline([
    # TODO: uzupełnij braki średnią
    # TODO: dodaj klasyfikator drzewa decyzyjnego
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
pipeline.fit(X_train, y_train)

print("Accuracy na zbiorze treningowym:", round(pipeline.score(X_train, y_train), 3))
print("Accuracy na zbiorze testowym:", round(pipeline.score(X_test, y_test), 3))


In [ ]:
# Siatka wartości hiperparametrów, które chcemy porównać.
params = {
    "tree__max_depth": # TODO: wybierz zakres,
    "tree__min_samples_leaf": # TODO: wybierz zakres,
}

# Dzielimy wyłącznie zbiór treningowy na trzy foldy.
# Używamy zwykłego KFold; dla klasyfikatora GridSearchCV domyślnie wybrałby StratifiedKFold.
cv_3 = KFold(n_splits=3, shuffle=True, random_state=42)

# GridSearchCV dopasuje pipeline dla każdej kombinacji parametrów i każdego foldu,
# a następnie wybierze konfigurację o najwyższej średniej accuracy w CV.
grid_pipeline = GridSearchCV(
    estimator=pipeline,
    param_grid=params,
    cv=cv_3,
    scoring="accuracy",
)


In [ ]:
grid_pipeline.fit(X_train, y_train)

selected_acc = grid_pipeline # TODO: oblicz accuracy z modelu z GridSearch (podpunkt b)) 
baseline_acc = pipeline # TODO: oblicz accuracy z modelu bez GridSearch (podpunkt a)) 
selected_auc = roc_auc_score(# TODO: oblicz AUC)
baseline_auc = roc_auc_score(# TODO: oblicz AUC)

print("Najlepsze parametry:", grid_pipeline.# TODO: wskaż najlepsze parametry z GridSearch)
print("Najlepsza accuracy w CV:", round(grid_pipeline.# TODO: wskaż wartość accuracy dla najlepszej konfiguracji hiperparametrów, 3))
print("Accuracy dostrojonego modelu na teście:", round(selected_acc, 3))
print("ROC AUC dostrojonego modelu na teście:", round(selected_auc, 3))
print("Accuracy bazowego modelu na teście:", round(baseline_acc, 3))
print("ROC AUC bazowego modelu na teście:", round(baseline_auc, 3))


### 2. Reprodukowalność eksperymentów

Losowy podział danych może zmienić nie tylko konkretne obserwacje w zbiorze testowym, ale też rozkład zmiennej docelowej i cech. Ustawienie `random_state` pozwala odtworzyć konkretny eksperyment.


#### Przykład: wpływ losowego podziału danych

Porównajmy kilka podziałów danych `titanic`. Ten sam `random_state` odtwarza identyczny zbiór testowy, ale różne wartości seeda prowadzą do innych obserwacji i nieco innych rozkładów zmiennych.



In [ ]:
reference_seed = 123
seeds = [123, 321, 999, 2026]
indices = np.arange(len(df))

# Ten sam seed zawsze odtwarza identyczny podział.
_, reference_test_idx = train_test_split(
    indices, test_size=0.3, random_state=reference_seed
)
_, repeated_test_idx = train_test_split(
    indices, test_size=0.3, random_state=reference_seed
)
print("Ten sam random_state daje identyczny test:",
      np.array_equal(reference_test_idx, repeated_test_idx))

rows = []
for seed in seeds:
    _, test_idx = train_test_split(indices, test_size=0.3, random_state=seed)
    test_data = df.iloc[test_idx]
    rows.append({
        "random_state": seed,
        "wspólne obserwacje z seedem 123 (%)": (
            100 * len(np.intersect1d(reference_test_idx, test_idx)) / len(reference_test_idx)
        ),
        "udział osób, które przeżyły": test_data["survived"].mean(),
        "średnia fare": test_data["fare"].mean(),
    })

comparison = pd.DataFrame(rows).round(3)
comparison

# Dla dwóch losowych testów o rozmiarze 30% oczekujemy około 30% wspólnych obserwacji.
comparison_other = comparison.loc[comparison["random_state"] != reference_seed]
full_survived_share = df["survived"].mean()
deviation_pp = 100 * (comparison["udział osób, które przeżyły"] - full_survived_share)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(comparison_other["random_state"].astype(str),
            comparison_other["wspólne obserwacje z seedem 123 (%)"])
axes[0].axhline(30, color="black", linestyle="--", label="wartość oczekiwana: 30%")
axes[0].set_xlabel("random_state")
axes[0].set_ylabel("Wspólne obserwacje (%)")
axes[0].set_title("Nakładanie się różnych zbiorów testowych")
axes[0].legend()

bars = axes[1].bar(comparison["random_state"].astype(str), deviation_pp)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("random_state")
axes[1].set_ylabel("Odchylenie udziału survived (p.p.)")
axes[1].set_title("Zmiana udziału klasy względem pełnego zbioru")
for bar, value in zip(bars, deviation_pp):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, value,
        f"{value:+.1f}", ha="center",
        va="bottom" if value >= 0 else "top",
    )

fig.tight_layout()
plt.show()


#### Wniosek

Różne seedy zmieniają większość obserwacji w zbiorze testowym, choć w tym przykładzie udział klas różni się tylko o kilka punktów procentowych. Dla silnie niezbalansowanej zmiennej docelowej takie wahania mogą być znacznie większe; później poznamy stratyfikację, która pomaga zachować podobny udział klas w częściach danych.

